# 00 — Data Quality Assessment (Khảo sát chất lượng dữ liệu)

**Bước ĐẦU TIÊN — phải chạy trước khi thiết kế pipeline.**

Mục đích: khảo sát dữ liệu thô theo **7 chiều chất lượng dữ liệu**, định lượng từng vấn đề và đề xuất
hành động. Chính các phát hiện ở đây là **cơ sở để định nghĩa luật** trong
`config/pipeline_config.json` (mục `data_quality`) và logic làm sạch ở tầng Silver.

| # | Chiều (Dimension) | Câu hỏi |
|---|---|---|
| 1 | Completeness | Có bao nhiêu giá trị rỗng/thiếu? |
| 2 | Uniqueness | Khóa nghiệp vụ có bị trùng? |
| 3 | Validity | Giá trị có đúng định dạng/kiểu/miền? |
| 4 | Consistency | Giá trị phân loại có nhất quán? |
| 5 | Accuracy / Range | Số có nằm trong biên hợp lý? |
| 6 | Structure | Cột bán cấu trúc (JSON) ra sao? |
| 7 | Timeliness / Temporal | Ngày giờ hợp lệ? Tỷ giá có phủ đủ các tháng? |

Đầu ra: bảng Delta **`dq_assessment_findings`** + báo cáo in trực tiếp. Notebook đọc **dữ liệu thô**
(`Files/raw`), không phụ thuộc tầng nào — nên đây thực sự là bước khảo sát đầu tiên.

> Bản local tương đương (pandas, chạy nhanh): `analysis/data_quality_assessment.py`.

In [1]:
pip install pandas

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/9.8 MB ? eta -:--:--
   --- ------------------------------------ 0.8/9.8 MB 12.4 MB/s eta 0:00:01
   ----------------------- ---------------- 5.8/9.8 MB 27.7 MB/s eta 0:00:01
   ---------------------------------------- 9.8/9.8 MB 23.4 MB/s  0:00:00
   ---------------------------------------- 0.0/12.3 MB ? eta -:--:--
   ----------------------------------- ---- 11.0/12.3 MB 53.8 MB/s eta 0:00:01
   ---------------------------------------- 12.3/12.3 MB 37.5 MB/s  0:00:00

   ---------------------------------------- 0/3 [tzdata]
   ---------------------------------------- 0/3 [tzdata]
   ---------------------------------------- 0/3 [tzdata]
   ---------------------------------------- 0/3 [tzdata]
   ---------------------------------------- 0/3 [tzdata]
   ---------------------------------------- 0/3 [tzdata]
   ------------- -------------------------- 1/3 [numpy]
 


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: C:\Users\LENOVO\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [ ]:
import json
from pyspark.sql import functions as F, types as T, Row

try:
    with open("/lakehouse/default/Files/config/pipeline_config.json", "r", encoding="utf-8") as fh:
        CONFIG = json.load(fh)
except FileNotFoundError:
    import notebookutils
    CONFIG = json.loads(notebookutils.fs.head("Files/config/pipeline_config.json", 5 * 1024 * 1024))

SALES_CSV = CONFIG["sources"]["sales"]["raw_file"]
FX_CSV = CONFIG["sources"]["exchange_rate"]["raw_file"]

# Đọc thô dạng chuỗi (giống nguyên tắc Bronze: không suy luận kiểu)
sales = (spark.read.option("header", "true").option("multiLine", "true")
         .option("quote", '"').option("escape", '"').option("inferSchema", "false").csv(SALES_CSV))
rates = spark.read.option("header", "true").option("inferSchema", "false").csv(FX_CSV)

N = sales.count()
NR = rates.count()
print(f"Phạm vi khảo sát: sales = {N} dòng, {len(sales.columns)} cột | exchange_rate = {NR} dòng")

# Tích lũy phát hiện: (dimension, table, column, issue, severity, count, pct, action, proposed_rule)
findings = []
def add(dim, tbl, col, issue, sev, cnt, action, rule=""):
    findings.append((dim, tbl, col, issue, sev, int(cnt), round(cnt / N * 100, 2), action, rule))

## 1) Completeness — tỉ lệ rỗng theo cột

In [ ]:
empty_exprs = [F.sum(F.when(F.trim(F.col(c)) == "", 1).otherwise(0)).alias(c) for c in sales.columns]
empties = sales.agg(*empty_exprs).collect()[0].asDict()
print("Số ô rỗng theo cột:")
for c, v in empties.items():
    if v:
        print(f"  {c:16s} {v:6d} ({v/N*100:5.1f}%)")
WARN_EMPTY = {"customer_age", "device_type", "discount_code", "feedback_score"}
for c, v in empties.items():
    if v:
        sev = "WARN" if c in WARN_EMPTY else "REJECT"
        add("Completeness", "sales", c, "Giá trị rỗng", sev, v,
            "Điền mặc định/null + cờ" if sev == "WARN" else "Cách ly nếu là cột bắt buộc")

## 2) Uniqueness — trùng khóa nghiệp vụ

In [ ]:
distinct_ids = sales.select("order_id").distinct().count()
dup = N - distinct_ids
print(f"order_id: {N} dòng / {distinct_ids} mã duy nhất -> {dup} dòng trùng")
add("Uniqueness", "sales", "order_id", "Trùng khóa nghiệp vụ", "REJECT", dup,
    "Giữ bản ghi đầu, cách ly bản trùng", "SAL_006 duplicate_order_id")

## 3) Validity — định dạng / kiểu / miền giá trị

In [ ]:
spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")
order_ts = F.coalesce(
    F.to_timestamp("order_date", "yyyy-MM-dd'T'HH:mm:ss.SSSSSS"),
    F.to_timestamp("order_date", "yyyy-MM-dd'T'HH:mm:ss"),
    F.to_timestamp("order_date", "dd/MM/yyyy HH:mm"),
    F.to_timestamp("order_date", "dd/MM/yyyy"))
amt = F.regexp_replace("total_amount", "[^0-9.\\-]", "").cast("double")
fb = F.col("feedback_score").cast("double")
v = sales.select(
    F.sum(F.when(F.col("order_date").rlike("^\\d{2}/\\d{2}/\\d{4}"), 1).otherwise(0)).alias("date_ddmmyyyy"),
    F.sum(F.when(order_ts.isNull(), 1).otherwise(0)).alias("date_unparseable"),
    F.sum(F.when(F.col("total_amount").contains("$"), 1).otherwise(0)).alias("amt_dollar"),
    F.sum(F.when(amt.isNull() | (amt <= 0), 1).otherwise(0)).alias("amt_bad"),
    F.sum(F.when((F.trim("feedback_score") == "") | ~fb.between(1, 5), 1).otherwise(0)).alias("fb_invalid"),
).collect()[0]
print(f"order_date dd/MM/yyyy: {v['date_ddmmyyyy']} | không parse: {v['date_unparseable']}")
print(f"total_amount có '$': {v['amt_dollar']} | <=0/không số: {v['amt_bad']}")
print(f"feedback_score ngoài 1-5 hoặc rỗng: {v['fb_invalid']}")
add("Validity", "sales", "order_date", "Nhiều định dạng ngày (ISO + dd/MM/yyyy)", "REJECT",
    v["date_ddmmyyyy"], "Parse đa định dạng; lỗi -> cách ly", "SAL_002 unparseable_order_date")
add("Validity", "sales", "total_amount", "Lẫn ký tự tiền tệ '$'", "CLEAN",
    v["amt_dollar"], "regexp_replace bỏ ký tự, ép double", "(làm sạch tại chỗ)")
add("Validity", "sales", "total_amount", "Giá trị <= 0 / không phải số", "REJECT",
    v["amt_bad"], "Cách ly", "SAL_003 invalid_total_amount")
add("Validity", "sales", "feedback_score", "Ngoài thang 1-5 (sentinel 99) hoặc rỗng", "WARN",
    v["fb_invalid"], "Giữ + cờ feedback_is_valid; lọc ở Gold", "SAL_101 feedback_out_of_scale")

## 4) Consistency — chuẩn hóa giá trị phân loại

In [ ]:
print("payment_method (các biến thể):")
sales.groupBy("payment_method").count().orderBy(F.desc("count")).show(truncate=False)
print("currency:"); sales.groupBy("currency").count().show()
print("order_status:"); sales.groupBy("order_status").count().show()
variants = sales.where(F.lower(F.regexp_replace(F.regexp_replace("payment_method", "_", ""), " ", "")) == "creditcard").count()
add("Consistency", "sales", "payment_method",
    "Một giá trị nhiều cách viết (credit_card/CREDITCARD/Credit Card)", "CLEAN",
    variants, "Chuẩn hóa về Credit Card/PayPal/COD", "(làm sạch tại chỗ)")

## 5) Accuracy / Range — biên giá trị số

In [ ]:
age = F.col("customer_age").cast("int")
ship = F.col("shipping_cost").cast("double")
a = sales.select(
    F.sum(F.when(ship < 0, 1).otherwise(0)).alias("ship_neg"),
    F.sum(F.when(age.isNotNull() & ~age.between(18, 100), 1).otherwise(0)).alias("age_out"),
    F.min(age).alias("age_min"), F.max(age).alias("age_max"),
).collect()[0]
print(f"shipping_cost < 0: {a['ship_neg']} | customer_age range [{a['age_min']},{a['age_max']}], ngoài 18-100: {a['age_out']}")
add("Accuracy", "sales", "shipping_cost", "Giá trị âm (bất khả thi)", "REJECT",
    a["ship_neg"], "Cách ly", "SAL_005 negative_or_invalid_shipping")
add("Accuracy", "sales", "customer_age", "Tuổi ngoài khoảng 18-100", "WARN",
    a["age_out"], "Giữ + cờ cảnh báo", "SAL_102 customer_age_out_of_range")

## 6) Structure — cột bán cấu trúc (JSON)

In [ ]:
ITEMS = T.ArrayType(T.StructType([
    T.StructField("product_id", T.StringType()), T.StructField("category", T.StringType()),
    T.StructField("price", T.DoubleType()), T.StructField("quantity", T.IntegerType())]))
CUST = T.StructType([T.StructField("name", T.StringType()), T.StructField("email", T.StringType()),
                     T.StructField("phone", T.StringType())])
items_p = F.from_json("items", ITEMS)
cust_p = F.from_json("customer_info", CUST)
s = sales.select(
    F.sum(F.when(items_p.isNull() | (F.size(items_p) == 0), 1).otherwise(0)).alias("items_bad"),
    F.sum(F.when(cust_p.isNull(), 1).otherwise(0)).alias("cust_bad"),
).collect()[0]
print(f"items JSON hỏng/rỗng: {s['items_bad']} | customer_info JSON hỏng: {s['cust_bad']}")
add("Structure", "sales", "items", "Mảng JSON hỏng/rỗng", "REJECT",
    s["items_bad"], "Cách ly", "SAL_004 invalid_items_json")
add("Structure", "sales", "customer_info", "JSON object lồng -> cần tách", "CLEAN",
    N - s["cust_bad"], "from_json tách name/email/phone", "(làm sạch tại chỗ)")

## 7) Timeliness / Temporal — ngày giờ & độ phủ tỷ giá theo tháng
Đây là phần khảo sát **ngày giờ** chuyên biệt: ngày có hợp lệ về thời gian không, và quan trọng nhất
là **tỷ giá có phủ đủ mọi tháng có đơn hàng không** — vì nếu thiếu, doanh thu VND sẽ bị null.

In [ ]:
ots = F.coalesce(
    F.to_timestamp("order_date", "yyyy-MM-dd'T'HH:mm:ss.SSSSSS"),
    F.to_timestamp("order_date", "yyyy-MM-dd'T'HH:mm:ss"),
    F.to_timestamp("order_date", "dd/MM/yyyy HH:mm"),
    F.to_timestamp("order_date", "dd/MM/yyyy"))
t = sales.select(
    F.sum(F.when(ots > F.current_timestamp(), 1).otherwise(0)).alias("future"),
    F.sum(F.when(ots.isNotNull() & ~F.year(ots).between(2024, 2025), 1).otherwise(0)).alias("outside"),
    F.min(ots).alias("dmin"), F.max(ots).alias("dmax")).collect()[0]
print(f"Phạm vi ngày: {t['dmin']} -> {t['dmax']}")
print(f"Ngày tương lai (> hôm nay): {t['future']} | ngoài cửa sổ 2024-2025: {t['outside']}")

# Độ phủ tỷ giá: mọi (năm, tháng) của đơn phải có dòng tỷ giá tương ứng
order_months = sales.select(F.year(ots).alias("y"), F.month(ots).alias("m")).distinct()
rate_months = rates.select(F.col("year").cast("int").alias("y"), F.col("month").cast("int").alias("m")).distinct()
uncovered = order_months.join(rate_months, ["y", "m"], "left_anti").count()
print(f"Tháng có đơn KHÔNG có tỷ giá (độ phủ chéo nguồn): {uncovered}")

add("Timeliness", "sales", "order_date", "Ngày trong tương lai (> hôm nay)", "REJECT",
    t["future"], "Cách ly (đơn không thể ở tương lai)", "SAL_105 order_date_in_future")
add("Timeliness", "sales", "order_date", "Ngày ngoài cửa sổ 2024-2025 (không có tỷ giá để quy đổi)", "WARN",
    t["outside"], "Giữ + cờ; cảnh báo quy đổi VND", "SAL_104 order_date_outside_rate_coverage")
add("Timeliness", "sales<->exchange_rate", "order year-month", "Tháng đơn thiếu tỷ giá để quy đổi VND",
    "REJECT" if uncovered else "OK", uncovered, "Kiểm tra ở Gold (left join -> đếm null)", "(Gold: missing_fx)")

## Exchange rate — kiểm tra nhanh

In [ ]:
fxchk = rates.select(
    F.sum(F.when(~F.col("year").cast("int").between(2000, 2100), 1).otherwise(0)).alias("bad_year"),
    F.sum(F.when(~F.col("month").cast("int").between(1, 12), 1).otherwise(0)).alias("bad_month"),
    F.sum(F.when(F.col("exchange_rate").cast("double") <= 0, 1).otherwise(0)).alias("bad_rate"),
).collect()[0]
print(f"exchange_rate: year lỗi={fxchk['bad_year']}, month lỗi={fxchk['bad_month']}, rate lỗi={fxchk['bad_rate']} (kỳ vọng 0/0/0)")

## Tổng hợp findings → ghi bảng Delta `dq_assessment_findings`

In [ ]:
schema = T.StructType([
    T.StructField("dimension", T.StringType()), T.StructField("table_name", T.StringType()),
    T.StructField("column_name", T.StringType()), T.StructField("issue", T.StringType()),
    T.StructField("severity", T.StringType()), T.StructField("affected_rows", T.LongType()),
    T.StructField("affected_pct", T.DoubleType()), T.StructField("recommended_action", T.StringType()),
    T.StructField("proposed_rule", T.StringType())])
dqdf = spark.createDataFrame([Row(*f) for f in findings], schema) \
            .withColumn("assessed_at", F.current_timestamp())
dqdf.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("dq_assessment_findings")

print("=== BẢNG PHÁT HIỆN CHẤT LƯỢNG (đã ghi vào dq_assessment_findings) ===")
dqdf.orderBy("severity", "dimension").show(50, truncate=60)
print("=== Tổng hợp theo mức độ ===")
dqdf.groupBy("severity").count().show()
print("KẾT LUẬN: các phát hiện REJECT/WARN ở trên là cơ sở định nghĩa luật trong\n"
      "config/pipeline_config.json (mục data_quality) và logic làm sạch ở tầng Silver (notebook 02).")